In [ ]:
%cd ..

import torch

import numpy as np

from env.dummy_envs import DummyEnvGraph
from policy.policies import GnnPpoPolicy
from traintest.rl_loop import rl_train

In [ ]:
# First: Testing the GNN Policy Training Loop on a basic env to ensure it runs without errors

seed = 70707
rng = np.random.default_rng(seed)

#create env, Graph observation Space, MultiDiscrete action space

graph_nodes = 6
node_feats = 5
edge_feats = 3
max_waypoints = 3
encode_sz = 3

env = DummyEnvGraph(n_nodes=graph_nodes, n_feats=node_feats, e_feats=edge_feats, max_wp=max_waypoints)
obs_space, act_space = env.get_spaces()

In [ ]:
#create policy

pol_name = 'p1'
core_hsize = 16
core_nbl = 2

encode_args = {'walk_length': encode_sz}
mp_args = {'in_channels': core_hsize, 'out_channels': core_hsize}
gnn_norm_args = {'in_channels': core_hsize}
gnn_act_args = {}
gnn_edge_drop_args = {'p': 0.05, 'force_undirected': True}

mpblock_args = {'mp_type': 'gcn', 'mp_args': mp_args, 'norm_type': 'batch', 'norm_args': gnn_norm_args,
                'act_type': 'relu', 'act_args': gnn_act_args, 'drop_graph': 'edge', 'drop_graph_args': gnn_edge_drop_args,
                'drop_x': True, 'dropx_p': 0.1, 'residual_node': True, 'edge_attr': False}

ff_hidden = 64

ffblock_args = {'in_size': core_hsize, 'out_size': core_hsize, 'hidden_size': ff_hidden, 'norm_type': 'batch',
                'norm_args': {'num_features': core_hsize}, 'act_type': 'relu', 'act_args': {}}

core_args = {'mp_args': mpblock_args, 'ff_args': ffblock_args}

policy_net_args = {'x_feats': node_feats, 'x_out': max_waypoints, 'core_hidden': core_hsize, 'core_blocks': core_nbl,
                    'core_args': core_args, 'encodings': 'rwpe', 'encode_size': encode_sz, 'encode_args': encode_args,
                    'global_pool': False}

value_net_args = {'x_feats': node_feats, 'x_out': 1, 'core_hidden': core_hsize, 'core_blocks': core_nbl,
                    'core_args': core_args, 'encodings': 'rwpe', 'encode_size': encode_sz, 'encode_args': encode_args,
                    'global_pool': True, 'gp_args': {'pool_type': 'sum'}}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


policy1 = GnnPpoPolicy(name=pol_name, obs_space=obs_space, act_space=act_space, policy_net_args=policy_net_args, 
                        value_net_args=value_net_args, out_graph_embed=False, device=device, rng=rng, n_steps=1024, batch_sz=32)

In [ ]:
#Execute the training loop proper 

rl_train(env=env, agent=policy1, alg='PPO')